# Stage 5 -- Interpretability and label robustness (E5, E6)
(1) Compares the LGC co-occurrence adjacency and the LAT label-label attention against
the empirical co-occurrence matrix (Spearman correlation + heatmaps).
(2) Label-noise robustness using the per-tool predictions: evaluation on the uncontested
subset and re-training on majority-vote labels.
**Accelerator: GPU.** **Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`,
`scvd-stage3-deep` (test predictions).

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/datasets/smnihalahmed/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage5_interpretability')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

2026-09-25 18:30:04,696 INFO stage stage5_interpretability initialised at /kaggle/working/stage5_interpretability
2026-09-25 18:30:04,697 INFO seeds=(0, 1, 2) quick=False


In [2]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

2026-09-25 18:30:29,418 INFO loaded DIVE: pre=(22330, 222) post=(22330, 177) labels=(22330, 9) tools=(26071, 49)
2026-09-25 18:30:29,420 INFO family code_metadata        6 features
2026-09-25 18:30:29,421 INFO family interface_source    46 features
2026-09-25 18:30:29,422 INFO family creation_opcodes   168 features
2026-09-25 18:30:29,423 INFO family tx_metadata         10 features
2026-09-25 18:30:29,423 INFO family temporal_position    8 features
2026-09-25 18:30:29,425 INFO family runtime_opcodes    158 features
2026-09-25 18:30:29,426 INFO view F_pre  220 features
2026-09-25 18:30:29,427 INFO view F_post 176 features
2026-09-25 18:30:29,428 INFO view F_all  396 features


In [3]:
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)
d = scvd.make_data(pre, post, lab, fam, 'all', *splits['S1'])
M = scvd.cooccurrence(d['y_tr'])
cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'tau': 0.2, 'batch': 256,
       'epochs': 60 if scvd.quick() else 120}
from scipy.stats import spearmanr
off = ~np.eye(8, dtype=bool)

# R2#5/#8 follow-up: this was a single un-varied run (SEEDS[0] only) -- GPU/cuDNN
# non-determinism means even a "fixed" seed gives a different trained model each
# time, so a single point estimate for the headline rho isn't defensible. Loop
# over all 3 seeds and report mean/range, same treatment as the robustness cell.
dep_rows = []
A_last = attn_last = None
for seed in SEEDS:
    scvd.set_seed(seed)
    lgc = scvd.LGC(d['fams'], M, dm=128, tau=0.2)
    _, _h = scvd.train_torch(lgc, d, cfg, log)
    A = scvd.lgc_A(M, 0.2)
    scvd.set_seed(seed)
    lat = scvd.LAT(d['fams'], dm=128, layers=3)
    _, _h = scvd.train_torch(lat, d, cfg, log)
    attn = scvd.extract_label_attention(lat, d)
    rho_lgc = spearmanr(M[off], A[off]).statistic
    rho_lat = spearmanr(M[off], attn[off]).statistic
    dep_rows.append({'seed': seed, 'rho_lgc': rho_lgc, 'rho_lat': rho_lat})
    log.info('seed%d: Spearman(M, A_lgc)=%.3f  Spearman(M, attn_lat)=%.3f', seed, rho_lgc, rho_lat)
    A_last, attn_last = A, attn   # keep one trained pair around for the pairwise CSV/plot below

dep_df = pd.DataFrame(dep_rows)
dep_df.to_csv(f'{root}/results/stage5_dependency_rho_by_seed.csv', index=False)
log.info('rho_lgc: mean=%.3f range=[%.3f,%.3f] | rho_lat: mean=%.3f range=[%.3f,%.3f]',
          dep_df['rho_lgc'].mean(), dep_df['rho_lgc'].min(), dep_df['rho_lgc'].max(),
          dep_df['rho_lat'].mean(), dep_df['rho_lat'].min(), dep_df['rho_lat'].max())

# per-pair CSV and plot still use one representative (last-seed) trained pair, as before
pd.DataFrame({'pair': [f'{scvd.LABEL_SHORT[i]}-{scvd.LABEL_SHORT[j]}'
                       for i in range(8) for j in range(8) if off[i, j]],
              'M': M[off], 'A_lgc': A_last[off], 'attn_lat': attn_last[off]}).to_csv(
    f'{root}/results/stage5_dependency_pairs.csv', index=False)
scvd.plot_dependency_maps(M, A_last, attn_last, root, log)

2026-09-25 18:30:52,214 INFO   ep 000 loss 2.4017 val_macro_f1 0.3842 *
2026-09-25 18:30:52,628 INFO   ep 001 loss 1.5038 val_macro_f1 0.3907 *
2026-09-25 18:30:53,005 INFO   ep 002 loss 2.0974 val_macro_f1 0.2988
2026-09-25 18:30:53,428 INFO   ep 003 loss 1.0621 val_macro_f1 0.4838 *
2026-09-25 18:30:53,807 INFO   ep 004 loss 0.8996 val_macro_f1 0.4163
2026-09-25 18:30:54,173 INFO   ep 005 loss 0.9119 val_macro_f1 0.5580 *
2026-09-25 18:30:54,565 INFO   ep 006 loss 0.7086 val_macro_f1 0.5089
2026-09-25 18:30:54,959 INFO   ep 007 loss 0.6563 val_macro_f1 0.5651 *
2026-09-25 18:30:55,354 INFO   ep 008 loss 0.8075 val_macro_f1 0.5016
2026-09-25 18:30:55,725 INFO   ep 009 loss 1.1201 val_macro_f1 0.6134 *
2026-09-25 18:30:56,107 INFO   ep 010 loss 1.7883 val_macro_f1 0.6332 *
2026-09-25 18:30:56,482 INFO   ep 011 loss 0.8484 val_macro_f1 0.6065
2026-09-25 18:30:56,858 INFO   ep 012 loss 0.7043 val_macro_f1 0.5688
2026-09-25 18:30:57,271 INFO   ep 013 loss 0.5732 val_macro_f1 0.6429 *
2026

In [4]:
# R2#8: head- and layer-resolved attention, checking whether averaging across
# heads/layers/samples (as in the headline rho figure) hides finer structure.
# Run on F_post, where the H5 label-conditioning exception was observed.
#
# Follow-up fix: the single SEEDS[0]-only run below did not replicate across kernel
# restarts (GPU/cuDNN non-determinism), so a specific head/layer's correlation is
# not a defensible finding on its own. Loop over all 3 seeds and only report a
# head/layer combo as notable if it's consistently non-trivial across seeds --
# otherwise the honest read is "no stable head/layer signal, average is representative."
d_post = scvd.make_data(pre, post, lab, fam, 'post', *splits['S1'])
M_post = scvd.cooccurrence(d_post['y_tr'])

head_rows = []
avg_rhos = []
for seed in SEEDS:
    scvd.set_seed(seed)
    lat_post = scvd.LAT(d_post['fams'], dm=128, layers=3)
    _, _h = scvd.train_torch(lat_post, d_post, cfg, log)

    A_full = scvd.extract_label_attention_full(lat_post, d_post)   # (layers, heads, 8, 8)
    n_layers, n_heads = A_full.shape[:2]
    for li in range(n_layers):
        for hi in range(n_heads):
            head_rows.append({'seed': seed, 'layer': li, 'head': hi,
                               'rho': spearmanr(M_post[off], A_full[li, hi][off]).statistic})

    avg_attn_post = scvd.extract_label_attention(lat_post, d_post)   # head/layer-averaged, for comparison
    rho_avg_post = spearmanr(M_post[off], avg_attn_post[off]).statistic
    avg_rhos.append(rho_avg_post)
    log.info('seed%d F_post attention: averaged rho=%.3f', seed, rho_avg_post)

head_df = pd.DataFrame(head_rows)
head_df.to_csv(f'{root}/results/stage5_attention_by_head_post.csv', index=False)

# per (layer, head): is the sign/magnitude consistent across all 3 seeds, or is it noise?
stab = (head_df.groupby(['layer', 'head'])['rho']
        .agg(['mean', 'min', 'max'])
        .assign(consistent_sign=lambda x: np.sign(x['min']) == np.sign(x['max']))
        .reset_index())
stab.to_csv(f'{root}/results/stage5_attention_by_head_post_stability.csv', index=False)
log.info('F_post attention: averaged rho mean=%.3f range=[%.3f,%.3f]',
          np.mean(avg_rhos), min(avg_rhos), max(avg_rhos))
log.info('head/layer cells with a consistent sign across all 3 seeds: %d of %d',
          int(stab['consistent_sign'].sum()), len(stab))
stab.sort_values('mean', ascending=False)


2026-09-25 18:39:17,591 INFO   ep 000 loss 1.4577 val_macro_f1 0.5525 *
2026-09-25 18:39:18,499 INFO   ep 001 loss 1.1200 val_macro_f1 0.6079 *
2026-09-25 18:39:19,446 INFO   ep 002 loss 1.4976 val_macro_f1 0.6473 *
2026-09-25 18:39:20,391 INFO   ep 003 loss 0.7964 val_macro_f1 0.6334
2026-09-25 18:39:21,306 INFO   ep 004 loss 0.7058 val_macro_f1 0.5848
2026-09-25 18:39:22,251 INFO   ep 005 loss 0.6358 val_macro_f1 0.6501 *
2026-09-25 18:39:23,143 INFO   ep 006 loss 0.7669 val_macro_f1 0.6475
2026-09-25 18:39:24,099 INFO   ep 007 loss 0.4399 val_macro_f1 0.6593 *
2026-09-25 18:39:25,011 INFO   ep 008 loss 0.8407 val_macro_f1 0.6444
2026-09-25 18:39:25,949 INFO   ep 009 loss 0.6542 val_macro_f1 0.6643 *
2026-09-25 18:39:26,879 INFO   ep 010 loss 0.7410 val_macro_f1 0.6693 *
2026-09-25 18:39:27,815 INFO   ep 011 loss 0.8271 val_macro_f1 0.6550
2026-09-25 18:39:28,794 INFO   ep 012 loss 0.7882 val_macro_f1 0.6839 *
2026-09-25 18:39:29,784 INFO   ep 013 loss 0.6186 val_macro_f1 0.6756
2026

,layer,head,mean,min,max,consistent_sign
1,0,1,0.080223,-0.005537,0.229528,False
6,1,2,-0.001139,-0.262474,0.145933,False
0,0,0,-0.040875,-0.132878,0.060287,False
10,2,2,-0.076304,-0.108749,-0.011552,True
3,0,3,-0.077102,-0.441490,0.132604,False
2,0,2,-0.091091,-0.165003,-0.001230,True
11,2,3,-0.122465,-0.393575,0.247437,False
7,1,3,-0.144931,-0.318592,0.082502,False
4,1,0,-0.169036,-0.330759,-0.013055,True
8,2,0,-0.232126,-0.362406,-0.149146,True


In [5]:
Y = lab.set_index('contractID')[scvd.LABELS]
tl = tools.set_index('contractID')
if tl.index.duplicated().any():
    tl = tl[~tl.index.duplicated(keep='first')]
tl = tl.reindex(Y.index)
tool_cols = {j: [c for c in tl.columns if c.endswith(f'_{j+1}')] for j in range(8)}
agree = pd.DataFrame(True, index=Y.index, columns=scvd.LABELS)
for j, col in enumerate(scvd.LABELS):
    tc = tl[tool_cols[j]]
    obs = tc.notna()
    vals = tc.fillna(-1).eq(Y[col].values[:, None] * 1.0).where(obs, True)
    agree[col] = vals.all(1)
uncontested = agree.all(1)
log.info('uncontested contracts: %d (%.1f%%)', uncontested.sum(), 100 * uncontested.mean())

# R2#4: class balance of the uncontested subset
unc_prev = Y.loc[uncontested].mean().to_dict()
unc_all_neg = float((Y.loc[uncontested].sum(1) == 0).mean())
log.info('uncontested per-label positive rate: %s', {k: round(v, 4) for k, v in unc_prev.items()})
log.info('uncontested all-negative fraction: %.4f', unc_all_neg)

maj = pd.DataFrame(np.nan, index=Y.index, columns=scvd.LABELS)
for j, col in enumerate(scvd.LABELS):
    tc = tl[tool_cols[j]]
    s = tc.sum(1); n = tc.notna().sum(1)
    maj[col] = np.where(n > 0, (s / n.clip(lower=1) > 0.5).astype(float), np.nan)
maj = maj.fillna(Y.astype(float))

res_runs = []
ids_te = d['ids_te']
for name, mdl_factory in [('LGC', lambda: scvd.LGC(d['fams'], M, dm=128, tau=0.2)),
                          ('LAT', lambda: scvd.LAT(d['fams'], dm=128, layers=3)),
                          ('BR-trunk', lambda: scvd.LAT(d['fams'], dm=128, layers=3, trunk_only=True))]:
    for seed in SEEDS:                                     # R1#2: was single-seed
        scvd.set_seed(seed)   # was np.random.seed(seed) -- didn't touch torch's RNG at all
        mdl = mdl_factory(); pt, _h = scvd.train_torch(mdl, d, cfg, log)
        th = scvd.tune_thresholds(d['y_va'], scvd.predict(mdl, d, 'va'))
        pred = (pt >= th).astype(int)
        yte = d['y_te']
        m_full = scvd.all_metrics(yte, pred, pt)

        mask = uncontested.reindex(ids_te).to_numpy()
        m_ag = scvd.all_metrics(yte[mask], pred[mask], pt[mask])

        ymaj = maj.reindex(ids_te)[scvd.LABELS].to_numpy(int)
        d2 = dict(d)
        d2['y_tr'] = maj.reindex(d['ids_tr'])[scvd.LABELS].to_numpy(int)
        d2['y_va'] = maj.reindex(d['ids_va'])[scvd.LABELS].to_numpy(int)
        d2['y_te'] = ymaj
        scvd.set_seed(seed)   # was np.random.seed(seed) -- didn't touch torch's RNG at all
        mdl2 = mdl_factory(); pt2, _h2 = scvd.train_torch(mdl2, d2, cfg, log)
        # R1#3: threshold-tune the majority-vote arm on its own held-out
        # validation split instead of a fixed 0.5 cut
        th2 = scvd.tune_thresholds(d2['y_va'], scvd.predict(mdl2, d2, 'va'))
        m_maj = scvd.all_metrics(ymaj, (pt2 >= th2).astype(int), pt2)

        for arm, m in [('full', m_full), ('agreement', m_ag), ('majority', m_maj)]:
            res_runs.append({'model': name, 'arm': arm, 'seed': seed,
                              'macro_f1': m['macro_f1'], 'macro_auc': m['macro_auc']})
        log.info('%-9s seed%d full=%.4f unc=%.4f maj(tuned)=%.4f | '
                 'full_auc=%.4f unc_auc=%.4f maj_auc=%.4f',
                 name, seed, m_full['macro_f1'], m_ag['macro_f1'], m_maj['macro_f1'],
                 m_full['macro_auc'], m_ag['macro_auc'], m_maj['macro_auc'])

res_df = pd.DataFrame(res_runs)
res_summary = (res_df.groupby(['model', 'arm'])[['macro_f1', 'macro_auc']]
               .agg(['mean', 'min', 'max']))
res_summary.columns = ['_'.join(c) for c in res_summary.columns]
res_summary = res_summary.reset_index()

res = {'per_seed': res_runs, 'summary': res_summary.to_dict(orient='records'),
       'uncontested_prevalence': unc_prev, 'uncontested_all_negative_fraction': unc_all_neg,
       'n_seeds': list(SEEDS)}
json.dump(res, open(f'{root}/results/stage5_robustness.json', 'w'), indent=1)
res_df.to_csv(f'{root}/results/stage5_robustness_per_seed.csv', index=False)

res_mean = {m: {arm: float(res_df[(res_df.model == m) & (res_df.arm == arm)]['macro_f1'].mean())
                for arm in ['full', 'agreement', 'majority']}
            for m in res_df['model'].unique()}
scvd.plot_robustness(res_mean, root, log)
res_summary


2026-09-25 18:44:43,493 INFO uncontested contracts: 2361 (10.6%)
2026-09-25 18:44:43,498 INFO uncontested per-label positive rate: {'Reentrancy': 0.0089, 'Access Control': 0.008, 'Arithmetic': 0.0021, 'Unchecked Return Values': 0.0025, 'DoS': 0.0021, 'Bad Randomness': 0.0008, 'Front Running': 0.0008, 'Time manipulation': 0.0034}
2026-09-25 18:44:43,498 INFO uncontested all-negative fraction: 0.9835
2026-09-25 18:44:43,991 INFO   ep 000 loss 2.4017 val_macro_f1 0.3842 *
2026-09-25 18:44:44,383 INFO   ep 001 loss 1.5038 val_macro_f1 0.3907 *
2026-09-25 18:44:44,774 INFO   ep 002 loss 2.0974 val_macro_f1 0.2988
2026-09-25 18:44:45,169 INFO   ep 003 loss 1.0621 val_macro_f1 0.4838 *
2026-09-25 18:44:45,602 INFO   ep 004 loss 0.8996 val_macro_f1 0.4163
2026-09-25 18:44:46,017 INFO   ep 005 loss 0.9119 val_macro_f1 0.5580 *
2026-09-25 18:44:46,455 INFO   ep 006 loss 0.7086 val_macro_f1 0.5089
2026-09-25 18:44:46,861 INFO   ep 007 loss 0.6563 val_macro_f1 0.5651 *
2026-09-25 18:44:47,261 INFO

,model,arm,macro_f1_mean,macro_f1_min,macro_f1_max,macro_auc_mean,macro_auc_min,macro_auc_max
0,BR-trunk,agreement,0.107578,0.072890,0.174123,0.871344,0.861798,0.877963
1,BR-trunk,full,0.794802,0.789983,0.800109,0.938852,0.938541,0.939349
2,BR-trunk,majority,0.543551,0.530556,0.557970,0.893643,0.890016,0.896350
3,LAT,agreement,0.171913,0.071735,0.278879,0.917352,0.901928,0.940082
4,LAT,full,0.797484,0.796117,0.798850,0.950092,0.948896,0.951825
5,LAT,majority,0.563483,0.559351,0.568067,0.893189,0.886481,0.898524
6,LGC,agreement,0.153791,0.091949,0.206164,0.841861,0.771259,0.929561
7,LGC,full,0.772863,0.764187,0.778567,0.922660,0.918876,0.925183
8,LGC,majority,0.543414,0.537275,0.554931,0.879926,0.869237,0.891088


In [7]:
title = 'SCVD Stage 5: interpretability and robustness'  # 46 chars, under the cap
url = scvd.push_dataset(root, 'stage5_interpretability', 'scvd-stage5-interpretability', title, log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)

Starting upload for file logs.zip


100%|██████████| 45.9k/45.9k [00:00<00:00, 126kB/s]


Upload successful: logs.zip (46KB)
Starting upload for file plots.zip


100%|██████████| 139k/139k [00:00<00:00, 372kB/s]


Upload successful: plots.zip (139KB)
Starting upload for file tables.zip


100%|██████████| 22.0/22.0 [00:00<00:00, 57.7B/s]


Upload successful: tables.zip (22B)
Starting upload for file results.zip


100%|██████████| 4.92k/4.92k [00:00<00:00, 13.4kB/s]


Upload successful: results.zip (5KB)
2026-09-25 19:18:45,426 INFO pushed new version -> https://www.kaggle.com/datasets/smnihalahmed/scvd-stage5-interpretability
DATASET: https://www.kaggle.com/datasets/smnihalahmed/scvd-stage5-interpretability


In [9]:
# ============================================================
# R2#3 circularity ablation -- standalone, no scvd.py edit needed
# ============================================================
import os, sys, json, numpy as np, pandas as pd

LIB = '/kaggle/input/datasets/smnihalahmed/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))
sys.path.insert(0, LIB)
import scvd

root, log = scvd.setup_stage('stage5_circularity')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS

pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)

# F_pre = code_metadata(6) + interface_source(46) + creation_opcodes(168)
# F_post = tx_metadata(10) + temporal_position(8) + runtime_opcodes(158)
# The two opcode families (326 of 396 features) are what a static analyzer itself
# parses to produce its verdict; tx_metadata/temporal_position are on-chain execution
# history none of the six tools examine. Drop the opcode families entirely and see
# whether curated-label macro-F1 survives on the 70 remaining, non-overlapping features.
EXCLUDE = ('creation_opcodes', 'runtime_opcodes')

def make_data_excluding(pre, post, lab, fam, exclude, ids_tr, ids_va, ids_te):
    """Like scvd.make_data(view='all') but with the named families dropped entirely,
    not just their columns zeroed -- zeroing would still count as a family token in
    LAT's architecture (self.k = len(fams)); this drops the key so self.k is correct."""
    keep = [n for n in (scvd.PRE_FAMS + scvd.POST_FAMS) if n not in exclude]
    vf = {n: fam[n] for n in keep}
    pre_need = [c for n in scvd.PRE_FAMS if n in vf for c in vf[n]]
    post_need = [c for n in scvd.POST_FAMS if n in vf for c in vf[n]]
    cats = {**scvd.CATS_PRE, **scvd.CATS_POST}
    src_ = {}
    if pre_need:
        src_["pre"] = {"df": pre.set_index("contractID")[pre_need], "cols": pre_need}
    if post_need:
        src_["post"] = {"df": post.set_index("contractID")[post_need], "cols": post_need}
    lab_i = lab.set_index("contractID")
    parts = {}
    for part, ids in (("tr", ids_tr), ("va", ids_va), ("te", ids_te)):
        s = set(ids)
        parts[part] = {k: v["df"][v["df"].index.isin(s)] for k, v in src_.items()}
    preps = {k: scvd.Prep(v["cols"], {c: cc for c, cc in cats.items() if c in v["cols"]})
             .fit(parts["tr"][k].reset_index()) for k, v in src_.items()}
    out = {"fams": vf}
    for part in ("tr", "va", "te"):
        A, offs = {}, {}
        blocks = []
        for k in src_:
            Ak = preps[k].transform(parts[part][k].reset_index())
            offs[k] = {c: j for j, c in enumerate(src_[k]["cols"])}
            A[k] = Ak; blocks.append(Ak)
        out["x_" + part] = {}
        for n in vf:
            k = "pre" if n in scvd.PRE_FAMS else "post"
            out["x_" + part][n] = A[k][:, [offs[k][c] for c in vf[n]]]
        out["X_" + part] = np.hstack(blocks)
        ids_p = parts[part][next(iter(src_))].index.to_numpy()
        out["ids_" + part] = ids_p
        out["y_" + part] = lab_i.loc[ids_p, scvd.LABELS].to_numpy(int)
    return out

d_noop = make_data_excluding(pre, post, lab, fam, EXCLUDE, *splits['S1'])
M_noop = scvd.cooccurrence(d_noop['y_tr'])
log.info('circularity ablation: %d features remain (of 396) -- families kept: %s',
         d_noop['X_tr'].shape[1], list(d_noop['fams']))

cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'tau': 0.2, 'batch': 256,
       'epochs': 60 if scvd.quick() else 120}

circ_rows = []
for name, mdl_factory in [('LGC', lambda: scvd.LGC(d_noop['fams'], M_noop, dm=128, tau=0.2)),
                          ('LAT', lambda: scvd.LAT(d_noop['fams'], dm=128, layers=3)),
                          ('BR-trunk', lambda: scvd.LAT(d_noop['fams'], dm=128, layers=3, trunk_only=True))]:
    for seed in SEEDS:
        scvd.set_seed(seed)
        mdl = mdl_factory(); pt, _h = scvd.train_torch(mdl, d_noop, cfg, log)
        th = scvd.tune_thresholds(d_noop['y_va'], scvd.predict(mdl, d_noop, 'va'))
        m = scvd.all_metrics(d_noop['y_te'], (pt >= th).astype(int), pt)
        circ_rows.append({'model': name, 'seed': seed, 'n_features': d_noop['X_tr'].shape[1],
                          'macro_f1': m['macro_f1'], 'macro_auc': m['macro_auc']})
        log.info('no-opcode %-9s seed%d macro_f1=%.4f macro_auc=%.4f',
                 name, seed, m['macro_f1'], m['macro_auc'])

circ_df = pd.DataFrame(circ_rows)
circ_df.to_csv(f'{root}/results/stage5_circularity_ablation.csv', index=False)
print(circ_df.groupby('model')[['macro_f1', 'macro_auc']].agg(['mean', 'min', 'max']).round(4))

2026-09-25 19:31:30,266 INFO stage stage5_circularity initialised at /kaggle/working/stage5_circularity
2026-09-25 19:31:31,426 INFO loaded DIVE: pre=(22330, 222) post=(22330, 177) labels=(22330, 9) tools=(26071, 49)
2026-09-25 19:31:32,353 INFO circularity ablation: 70 features remain (of 396) -- families kept: ['code_metadata', 'interface_source', 'tx_metadata', 'temporal_position']
2026-09-25 19:31:32,750 INFO   ep 000 loss 3.5213 val_macro_f1 0.2164 *
2026-09-25 19:31:33,114 INFO   ep 001 loss 1.5117 val_macro_f1 0.2508 *
2026-09-25 19:31:33,477 INFO   ep 002 loss 2.4052 val_macro_f1 0.2960 *
2026-09-25 19:31:33,843 INFO   ep 003 loss 1.2241 val_macro_f1 0.4659 *
2026-09-25 19:31:34,217 INFO   ep 004 loss 0.8811 val_macro_f1 0.4685 *
2026-09-25 19:31:34,581 INFO   ep 005 loss 1.3700 val_macro_f1 0.4174
2026-09-25 19:31:34,937 INFO   ep 006 loss 0.9158 val_macro_f1 0.4374
2026-09-25 19:31:35,319 INFO   ep 007 loss 0.8324 val_macro_f1 0.5309 *
2026-09-25 19:31:35,712 INFO   ep 008 lo